In [2]:
import random
from pathlib import Path
from statistics import mean

import numpy as np
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch import nn

from tqdm import tqdm
import timm
from pytorch_datasets.ISIC2018 import ISIC2018
from pytorch_datasets.BreakHis import BreakHis
from pytorch_datasets.augmentations import Cutout, mixup, mixup_criterion, cutmix, cutmix_criterion

from easyfsl.samplers import TaskSampler
from easyfsl.methods.utils import evaluate
from torch.utils.data import DataLoader

In [3]:
random_seed = 0
np.random.seed(random_seed)
torch.manual_seed(random_seed)
random.seed(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Utils

In [4]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Hyperparameters

In [5]:
# Hyper-Parameters
learning_rate       = 1e-6

n_evaluation_tasks  = 600
n_epochs            = 100
n_tasks_per_epoch   = 2000

In [6]:
meta_lr    = 1
inner_lr    = 0.001
meta_iters  = 100
eval_iters  = 5
inner_iters = 5
eval_interval = 1

eval_batch_size = 10

In [7]:
n_way   = 2
n_shot  = 3
n_query = 10

n_episodes          = 40000
n_tasks_per_batch   = 10
n_validation_tasks  = 400

DEVICE = "cuda"
n_workers = 4

datasets  = ['ISIC2018', 'BreakHis']
dataset   = 0

if dataset == 0:
  root  = 'C:/Datasets/ISIC2018'
  train_set = ISIC2018(root=root, split='train', training=True)
  val_set   = ISIC2018(root=root, split='validation', training=False)

if dataset == 1:
  root          = 'E:/Datasets/BreaKHis_v1'
  magnification = 100

  train_set = BreakHis(root=root, magnification=magnification, split='train', training=True)
  val_set   = BreakHis(root=root, magnification=magnification, split='validation', training=False)


train_sampler = TaskSampler(
    train_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_tasks_per_batch
)
val_sampler = TaskSampler(
    val_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=eval_batch_size
)

# Finally, the DataLoader. We customize the collate_fn so that batches are delivered
# in the shape: (support_images, support_labels, query_images, query_labels, class_ids)
train_loader = DataLoader(
    train_set,
    batch_sampler=train_sampler,
    num_workers=n_workers,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=train_sampler.episodic_collate_fn,
)
val_loader = DataLoader(
    val_set,
    batch_sampler=val_sampler,
    num_workers=n_workers,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=val_sampler.episodic_collate_fn,
)

In [8]:
x = next(iter(train_loader))

In [14]:
xm = next(iter(train_loader))

In [13]:
x[4]

[1, 5]

In [17]:
xm[4]

[0, 2]

In [ ]:
mixup

### Define Model

In [29]:
from easyfsl.methods import PrototypicalNetworks, FewShotClassifier, RelationNetworks, MatchingNetworks
from easyfsl.modules import resnet12

model_name = 'vit_small_patch16_224'

vit = timm.create_model(model_name, pretrained=True, in_chans=3, num_classes=2)
# vit.head = nn.Identity() # vit
# vit.head.fc = nn.Identity() # swin
# vit.fc = nn.Identity() # resnet50

convolutional_network = resnet12()
few_shot_classifier = PrototypicalNetworks(vit).to(DEVICE)

In [8]:
len(list(vit.parameters()))

152

In [9]:
count_parameters(vit)

21666434

In [10]:
from torch.optim import SGD, Optimizer
from torch.optim.lr_scheduler import MultiStepLR, LinearLR, SequentialLR
from torch.utils.tensorboard import SummaryWriter


LOSS_FUNCTION = nn.CrossEntropyLoss()

n_epochs = 100
scheduler_milestones = [120, 160]
scheduler_gamma = 0.1
learning_rate = 1e-5
tb_logs_dir = Path(".")

train_optimizer = SGD(
    few_shot_classifier.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4
)

warmup_scheduler = LinearLR(
    train_optimizer,
    start_factor=0.02,
    end_factor=1, 
    total_iters=5
)
cosine_scheduler = CosineAnnealingLR(
    train_optimizer,
    T_max=n_epochs-5,
    eta_min=1e-6
)

seq_scheduler = SequentialLR(
    train_optimizer, 
    [warmup_scheduler, cosine_scheduler],
    milestones=[5]
)


train_scheduler = MultiStepLR(
    train_optimizer,
    milestones=scheduler_milestones,
    gamma=scheduler_gamma,
)

train_scheduler = seq_scheduler
# tb_writer = SummaryWriter(log_dir=str(tb_logs_dir))

## Training

In [11]:
import copy

def inner_loop(
    model, data_loader: DataLoader, criterion: nn.modules.loss, inner_iters: int, inner_lr: float, meta_lr
):
    for (
        support_images,
        support_labels,
        query_images,
        query_labels,
        _,
    ) in data_loader:
        task_model = copy.deepcopy(model).cuda()
        task_optimizer = optim.SGD(task_model.parameters(), lr=inner_lr)
        
        task_model.train()

        for _ in range(inner_iters):
            data, target = support_images.to(DEVICE), support_labels.to(DEVICE)
            task_optimizer.zero_grad()
            output = task_model(data)
            loss = criterion(output, target)
            loss.backward()
            task_optimizer.step()
                
        for (task_param, initial_param) in zip(task_model.parameters(), model.parameters()):
            meta_gradient = (task_param.data - initial_param.data) / inner_lr
            # meta_gradients.append(meta_gradient)

            initial_param.data += meta_lr * meta_gradient.data / n_tasks_per_batch

In [12]:
def reptile_evaluate(
    model, data_loaders, criterion: nn.modules.loss, inner_iters: int, inner_lr: float
):
    accuracies = []

    for data_loader in data_loaders:
        with tqdm(
            enumerate(data_loader), total=len(data_loader), desc=" eval loop", position=1, leave=False,
        ) as tqdm_train:
            for episode_index, (
                support_images,
                support_labels,
                query_images,
                query_labels,
                _,
            ) in tqdm_train:
                task_model = copy.deepcopy(model).cuda()
                task_optimizer = optim.SGD(task_model.parameters(), lr=inner_lr)
                
                task_model.train()

                for _ in range(inner_iters):
                    data, target = support_images.to(DEVICE), support_labels.to(DEVICE)
                    task_optimizer.zero_grad()
                    output = task_model(data)
                    loss = criterion(output, target)
                    loss.backward()
                    task_optimizer.step()
                
                query_preds = model(query_images.to(DEVICE))
                query_preds = query_preds.argmax(axis=1).cpu().numpy()

                num_correct = (query_preds == query_labels.numpy()).sum()

                accuracies.append(num_correct / len(query_labels))

    return accuracies

In [24]:
def reptile(
    model,
    train_loader,
    test_loader,
    criterion,
    meta_iters,
    inner_iters,
    eval_iters,
    meta_lr,
    inner_lr,
    eval_interval,
):
    
    training = []
    testing = []

    with tqdm(range(meta_iters), position=0, desc=" outer",) as reptile_tqdm:
        for meta_iter in reptile_tqdm:
            frac_done = meta_iter / meta_iters
            cur_meta_lr = (1 - frac_done) * meta_lr

            meta_gradients = []

            inner_loop(
                model           = model, 
                data_loader     = train_loader, 
                criterion       = criterion, 
                inner_steps     = inner_iters, 
                inner_lr        = inner_lr, 
                meta_lr         = cur_meta_lr
            )

            # for idx, param in enumerate(vit.parameters()):
            #     param.data += cur_meta_lr * meta_gradients[idx].data / n_tasks_per_batch
            
            if meta_iter % eval_interval == 0:
                accuracies = reptile_evaluate(model=model, data_loaders=[train_loader, test_loader], criterion=criterion, inner_iters=eval_iters, inner_lr=inner_lr,)
                training.append(accuracies[0])
                testing.append(accuracies[1])
                if meta_iter % 1 == 0:
                    reptile_tqdm.set_postfix(train_accuracy=accuracies[0], test_accuracy=accuracies[1])

In [25]:
reptile(
    model=vit,
    train_loader=train_loader,
    test_loader=val_loader,
    criterion=LOSS_FUNCTION,
    meta_iters=meta_iters,
    inner_iters=inner_iters,
    eval_iters=eval_iters,
    meta_lr=meta_lr,
    inner_lr=inner_lr,
    eval_interval=1,
)

 outer: 100%|██████████| 100/100 [12:52<00:00,  7.72s/it, test_accuracy=0.95, train_accuracy=0.7]


In [14]:
save = copy.deepcopy(vit)

In [15]:
# torch.save(save, 'save.pth')

In [41]:
few_shot_classifier.load_state_dict(best_state)

<All keys matched successfully>

## Evaluation

In [26]:
def evaluate(
    model, data_loader, criterion: nn.modules.loss, inner_iters: int, inner_lr: float
):
    accuracies = []

    with tqdm(
        enumerate(data_loader), total=len(data_loader), desc=" eval loop",
    ) as tqdm_train:
        for episode_index, (
            support_images,
            support_labels,
            query_images,
            query_labels,
            _,
        ) in tqdm_train:
            task_model = copy.deepcopy(model).cuda()
            task_optimizer = optim.SGD(task_model.parameters(), lr=inner_lr)
            
            task_model.train()

            for _ in range(inner_iters):
                data, target = support_images.to(DEVICE), support_labels.to(DEVICE)
                task_optimizer.zero_grad()
                output = task_model(data)
                loss = criterion(output, target)
                loss.backward()
                task_optimizer.step()
            
            query_preds = model(query_images.to(DEVICE))
            query_preds = query_preds.argmax(axis=1).cpu().numpy()

            num_correct = (query_preds == query_labels.numpy()).sum()

            accuracies.append(num_correct / len(query_labels))

    return np.mean(accuracies)

In [27]:
n_validation_tasks  = 400

val_sampler = TaskSampler(
        val_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_validation_tasks
    )

    # Finally, the DataLoader. We customize the collate_fn so that batches are delivered
    # in the shape: (support_images, support_labels, query_images, query_labels, class_ids)
test_loader = DataLoader(
    val_set,
    batch_sampler=val_sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=val_sampler.episodic_collate_fn,
)

In [30]:
accuracy = evaluate(vit, test_loader, criterion=LOSS_FUNCTION, inner_iters=inner_iters, inner_lr=inner_lr)
print(f"Average accuracy : {(100 * accuracy):.2f} %")

 eval loop: 100%|██████████| 400/400 [01:56<00:00,  3.43it/s]

Average accuracy : 50.11 %


In [47]:
#Evaluation Loop
ways   = [2, 3]
shots  = [3, 5, 10]
n_query = 10

n_validation_tasks  = 400

n_workers = 8

for way in ways:
  for shot in shots:
    val_sampler = TaskSampler(
        val_set, n_way=way, n_shot=shot, n_query=n_query, n_tasks=n_validation_tasks
    )

    # Finally, the DataLoader. We customize the collate_fn so that batches are delivered
    # in the shape: (support_images, support_labels, query_images, query_labels, class_ids)
    test_loader = DataLoader(
        val_set,
        batch_sampler=val_sampler,
        num_workers=n_workers,
        pin_memory=True,
        persistent_workers=True,
        collate_fn=val_sampler.episodic_collate_fn,
    )

    accuracy = evaluate(few_shot_classifier, val_loader, device=DEVICE)
    print(f"Average accuracy : {(100 * accuracy):.2f} %")

  0%|          | 0/400 [00:05<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 24952, 17284, 21592, 30680, 25576, 10652, 22200, 31128) exited unexpectedly

In [ ]:
n_test_tasks = 1000

test_set = CUB(split="test", training=False)
test_sampler = TaskSampler(
    test_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_test_tasks
)
test_loader = DataLoader(
    test_set,
    batch_sampler=test_sampler,
    num_workers=n_workers,
    pin_memory=True,
    collate_fn=test_sampler.episodic_collate_fn,
)

NameError: ignored

In [17]:
accuracy = evaluate(few_shot_classifier, val_loader, device=DEVICE)
print(f"Average accuracy : {(100 * accuracy):.2f} %")

100%|██████████| 400/400 [01:13<00:00,  5.41it/s, accuracy=0.66] 

Average accuracy : 65.96 %


In [18]:
torch.save(few_shot_classifier, f'E:/Models/protonet_checkpoints/{model_name}.pt')